# BEB-La-DII: Phase 1 Training (Awakening)
Этот блокнот предназначен для запуска первой фазы дистилляции в среде Kaggle. Он включает автоматическую настройку путей и безопасную установку зависимостей.

In [ ]:
# 1. БЕЗОПАСНАЯ УСТАНОВКА ЗАВИСИМОСТЕЙ
import subprocess
import sys

def install_packages():
    # Фиксируем версии из pyproject.toml для стабильности
    packages = [
        "transformers==4.57.2",
        "indexed-parquet-dataset",
        "optimum-intel[openvino]",
        "wandb",
        "accelerate"
    ]
    print("--- Проверка и установка пакетов ---")
    for pkg in packages:
        # Устанавливаем БЕЗ --upgrade, чтобы не сломать системные пакеты Kaggle
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", pkg])
        print(f"Установлено/Проверено: {pkg}")

install_packages()

# 2. НАСТРОЙКА ПУТЕЙ ИМПОРТА
from pathlib import Path

current_path = Path.cwd()
project_root = None

for path in [current_path] + list(current_path.parents):
    if (path / "src").exists():
        project_root = path
        break

if project_root:
    root_str = str(project_root.absolute())
    print(f"Корень проекта найден: {root_str}")
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
else:
    print("WARNING: Директория 'src' не найдена. Если вы клонировали репозиторий, убедитесь, что блокнот находится внутри него.")

In [ ]:
import os
import torch
from torch.optim import AdamW
from tqdm.auto import tqdm
import wandb
import shutil

# Импорты проекта
from src.beb_la_dii.model.distiller import ReasoningDistiller
from src.beb_la_dii.model.assembler import ModelAssembler
from src.beb_la_dii.utils.tokenizer import get_tokenizer
from src.beb_la_dii.utils.loss import DistillationLoss
from src.beb_la_dii.utils.data import get_dataloader
from src.beb_la_dii.utils.experiment_tracker import ExperimentTracker

print("Библиотеки и модули проекта успешно импортированы.")

In [ ]:
# КОНСТАНТЫ И ГИПЕРПАРАМЕТРЫ
BASE_MODEL_NAME = "answerdotai/ModernBERT-large"
TEACHER_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
EPOCHS = 1
LEARNING_RATE = 5e-5
STAGE = 'awakening' # 'awakening' для Stage 1, 'reasoning' для Stage 2
VERSION = "v1.0"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
def setup_kaggle():
    """Настройка путей для Kaggle: симлинки данных из Input в ./data/"""
    if not os.path.exists("/kaggle/input"):
        return None
        
    print("Настройка путей Kaggle...")
    os.makedirs("data", exist_ok=True)
    
    input_base = "/kaggle/input"
    resource_ds = None
    
    for ds_name in os.listdir(input_base):
        if "bebladii" in ds_name.lower():
            resource_ds = os.path.join(input_base, ds_name)
            break
            
    if not resource_ds:
        print("WARN: Датасет bebladii-resources не найден в /kaggle/input")
        return None

    ds_data_path = os.path.join(resource_ds, "data")
    if os.path.exists(ds_data_path):
        for folder in os.listdir(ds_data_path):
            src = os.path.join(ds_data_path, folder)
            dst = os.path.join("data", folder)
            if not os.path.exists(dst):
                print(f"Создание симлинка: {dst} -> {src}")
                os.symlink(src, dst)
    
    return resource_ds

resource_ds = setup_kaggle()

In [ ]:
# ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Инициализация системы на {device}...")

alt_roots = [resource_ds] if resource_ds else None
assembler = ModelAssembler(alt_roots=alt_roots)

distiller = assembler.assemble_phase1_distiller(
    teacher_id=TEACHER_NAME,
    version=VERSION,
    device_map="auto"
)

hyperparams = {
    "base_model": BASE_MODEL_NAME,
    "teacher": TEACHER_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "target_layers": 40,
    "version": VERSION
}

tracker = ExperimentTracker(project_root=".", stage=STAGE)

In [ ]:
# НАСТРОЙКА ГРАДИЕНТОВ (FREEZE)
for param in distiller.parameters():
    param.requires_grad = False
    
for param in distiller.student.parameters():
    param.requires_grad = True
    
for param in distiller.input_projector.parameters():
    param.requires_grad = True
    
for proj in distiller.feature_projectors.values():
    for param in proj.parameters():
        param.requires_grad = True
        
print(f"Обучаемых параметров: {sum(p.numel() for p in distiller.parameters() if p.requires_grad):,}")

In [ ]:
# ПОДГОТОВКА ДАННЫХ
print(f"Загрузка данных для стадии: {STAGE}...")
dataloader = get_dataloader(stage=STAGE, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)

dataset_metadata = []
for entry in dataloader.dataset.index_map:
    ds = entry['ds']
    ds_info = {
        "type": entry['type'],
        "count": entry['end'] - entry['start']
    }
    dataset_metadata.append(ds_info)

tracker.log_metadata(hyperparams, dataset_metadata)
print(f"Данные готовы. Всего батчей: {len(dataloader)}")

In [ ]:
# ТРЕНИРОВОЧНЫЙ ЦИКЛ
optimizer = AdamW(filter(lambda p: p.requires_grad, distiller.parameters()), lr=LEARNING_RATE)
criterion = DistillationLoss()

if os.environ.get("WANDB_API_KEY"):
    wandb.init(project="beb-la-dii-phase1", name=f"latentbert-{STAGE}-{VERSION}")
else:
    print("WANDB_API_KEY не найден. Логирование в WandB отключено.")

distiller.train()
progress_bar = tqdm(dataloader, desc=f"Training Phase 1 ({STAGE})")

accum_loss = 0.0
optimizer.zero_grad()

for step, batch in enumerate(progress_bar):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    
    student_states, teacher_targets = distiller(input_ids, attention_mask)
    loss = criterion(student_states, teacher_targets) / GRAD_ACCUM_STEPS
    
    loss.backward()
    accum_loss += loss.item()
    
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(distiller.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        avg_loss = accum_loss * GRAD_ACCUM_STEPS
        progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
        
        metrics = {"loss": avg_loss, "step": step, "lr": LEARNING_RATE}
        if wandb.run: wandb.log(metrics)
        tracker.log_step(step, metrics)
        accum_loss = 0.0

In [ ]:
# СОХРАНЕНИЕ
print("Сохранение финальных результатов...")
state_to_save = {
    "latentBERT_state_dict": distiller.student.state_dict(),
    "input_projector": distiller.input_projector.state_dict(),
    "feature_projectors": distiller.feature_projectors.state_dict(),
    "config": hyperparams
}

tracker.save_checkpoint(
    state_to_save, 
    optimizer_state=optimizer.state_dict(), 
    name=f"latentbert_{STAGE}_final"
)

tracker.finish()
if wandb.run: wandb.finish()
print("Дистилляция завершена успешно!")